# 📈 SBER 5-Minute Intraday Research Pipeline

**Goal:** feature engineering on raw Finam OHLCV candles → XGBoost classifier → probability calibration → feature importance.

**Target variable:** `target_is_green_next = 1` if next 5m candle closes above its open, else 0.

**Important:** all features use only current-bar-close or past-bar data. No look-ahead bias.

## Cell 1 — Imports & Display Settings

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pandas import DataFrame, concat
from IPython.display import display

import pandas_ta as ta

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (15, 5)

import sklearn
from packaging.version import Version
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss,
    brier_score_loss, classification_report, confusion_matrix,
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

# ── sklearn version compatibility ─────────────────────────────────────────
_SKLEARN_VERSION = Version(sklearn.__version__)
_CALIB_ESTIMATOR_KWARG = 'estimator' if _SKLEARN_VERSION >= Version('1.2') else 'base_estimator'

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('sklearn:', sklearn.__version__)
print('xgboost:', __import__('xgboost').__version__)
print(f'CalibratedClassifierCV kwarg: "{_CALIB_ESTIMATOR_KWARG}"')
print('All imports OK ✓')

## Cell 2 — Helper: series_to_supervised

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = DataFrame(data)
    col_list = data.columns.tolist()
    cols, names = list(), list()
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('%s_%d(t-%d)' % (col_list[j], j+1, i)) for j in range(n_vars)]
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('%s_var%d(t)' % (col_list[j], j+1)) for j in range(n_vars)]
        else:
            names += [('%s_var%d(t+%d)' % (col_list[j], j+1, i)) for j in range(n_vars)]
    agg = concat(cols, axis=1)
    agg.columns = names
    if dropnan:
        agg.dropna(inplace=True)
    return agg

print('series_to_supervised defined ✓')

## Cell 3 — Feature Engineering Functions

In [ ]:
EPS = 1e-12
DATA_PATH = './Сбербанк/year_result.csv'

def prepare_ohlcv_dataframe(df):
    required = {'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    out = df.copy(deep=True)
    for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    if {'DATE', 'TIME'}.issubset(out.columns):
        date_str = out['DATE'].astype(str).str.strip()
        time_str = out['TIME'].astype(str).str.zfill(6).str.strip()
        out['datetime'] = pd.to_datetime(date_str + time_str, format='%Y%m%d%H%M%S', errors='coerce')
        out = out.sort_values('datetime', kind='stable').reset_index(drop=True)
    return out

def add_domain_features(df):
    out = df.copy(deep=True)
    rng = out['HIGH'] - out['LOW']
    safe_rng = rng.where(rng.abs() > EPS, np.nan)
    max_oc = np.maximum(out['OPEN'], out['CLOSE'])
    min_oc = np.minimum(out['OPEN'], out['CLOSE'])
    prev_close = out['CLOSE'].shift(1)
    prev_high = out['HIGH'].shift(1)
    prev_low = out['LOW'].shift(1)
    out['candle_body'] = out['CLOSE'] - out['OPEN']
    out['body_abs'] = out['candle_body'].abs()
    out['candle_range'] = rng
    out['upper_wick'] = out['HIGH'] - max_oc
    out['lower_wick'] = min_oc - out['LOW']
    out['body_to_range'] = (out['body_abs'] / safe_rng).clip(0, 1)
    out['upper_wick_to_range'] = (out['upper_wick'] / safe_rng).clip(0, 1)
    out['lower_wick_to_range'] = (out['lower_wick'] / safe_rng).clip(0, 1)
    out['close_pos_in_range'] = ((out['CLOSE'] - out['LOW']) / safe_rng).clip(0, 1)
    out['open_pos_in_range'] = ((out['OPEN'] - out['LOW']) / safe_rng).clip(0, 1)
    out['direction'] = np.sign(out['candle_body']).astype('float64')
    out['is_green'] = (out['CLOSE'] > out['OPEN']).astype('int8')
    out['is_red'] = (out['CLOSE'] < out['OPEN']).astype('int8')
    out['is_doji_like'] = (out['body_to_range'] <= 0.1).astype('int8')
    out['ret_1'] = out['CLOSE'].pct_change(1)
    out['gap_from_prev_close'] = out['OPEN'] / prev_close - 1.0
    out['close_to_prev_high'] = out['CLOSE'] / prev_high - 1.0
    out['close_to_prev_low'] = out['CLOSE'] / prev_low - 1.0
    out['range_pct_close'] = out['candle_range'] / out['CLOSE'].replace(0, np.nan)
    out['body_x_vol'] = out['body_abs'] * out['VOL']
    out['signed_vol'] = out['direction'] * out['VOL']
    out['money_flow_proxy'] = ((out['CLOSE'] - out['LOW']) - (out['HIGH'] - out['CLOSE'])) / safe_rng * out['VOL']
    return out

def add_rolling_features(df, windows=(6, 12, 24)):
    out = df.copy(deep=True)
    for w in windows:
        out[f'ret_{w}'] = out['CLOSE'].pct_change(w)
        rng_mean = out['candle_range'].rolling(w, min_periods=w).mean()
        rng_std = out['candle_range'].rolling(w, min_periods=w).std()
        out[f'range_mean_{w}'] = rng_mean
        out[f'range_std_{w}'] = rng_std
        out[f'range_zscore_{w}'] = (out['candle_range'] - rng_mean) / rng_std.replace(0, np.nan)
        out[f'body_mean_{w}'] = out['body_abs'].rolling(w, min_periods=w).mean()
        close_ma = out['CLOSE'].rolling(w, min_periods=w).mean()
        close_std = out['CLOSE'].rolling(w, min_periods=w).std()
        out[f'close_ma_{w}'] = close_ma
        out[f'close_vs_ma_{w}'] = out['CLOSE'] / close_ma - 1.0
        out[f'close_zscore_{w}'] = (out['CLOSE'] - close_ma) / close_std.replace(0, np.nan)
        vol_ma = out['VOL'].rolling(w, min_periods=w).mean()
        out[f'vol_ma_{w}'] = vol_ma
        out[f'vol_ratio_{w}'] = out['VOL'] / vol_ma.replace(0, np.nan)
    return out

def add_calendar_features(df):
    if 'datetime' not in df.columns:
        raise ValueError("Column 'datetime' not found. Run prepare_ohlcv_dataframe first.")
    out = df.copy(deep=True)
    dt = out['datetime']
    out['hour'] = dt.dt.hour
    out['minute'] = dt.dt.minute
    out['dayofweek'] = dt.dt.dayofweek
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 5)
    out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 5)
    out['minutes_from_session_open'] = out['hour'] * 60 + out['minute'] - 600
    out['is_opening_30m'] = (out['minutes_from_session_open'] < 30).astype('int8')
    out['is_first_hour'] = (out['minutes_from_session_open'] < 60).astype('int8')
    out['date_only'] = dt.dt.date
    out['bar_in_day'] = out.groupby('date_only').cumcount()
    out['is_first_bar_of_day'] = (out['bar_in_day'] == 0).astype('int8')
    out = out.drop(columns=['date_only'])
    return out

def add_ta_features(df):
    out = df.copy(deep=True)
    out['ema_10'] = ta.ema(out['CLOSE'], length=10)
    out['ema_20'] = ta.ema(out['CLOSE'], length=20)
    out['sma_20'] = ta.sma(out['CLOSE'], length=20)
    out['rsi_14'] = ta.rsi(out['CLOSE'], length=14)
    out['atr_14'] = ta.atr(out['HIGH'], out['LOW'], out['CLOSE'], length=14)
    out['natr_14'] = ta.natr(out['HIGH'], out['LOW'], out['CLOSE'], length=14)
    macd = ta.macd(out['CLOSE'], fast=12, slow=26, signal=9)
    if macd is not None:
        out = pd.concat([out, macd], axis=1)
    bbands = ta.bbands(out['CLOSE'], length=20)
    if bbands is not None:
        out = pd.concat([out, bbands], axis=1)
    bbl = 'BBL_20_2.0'
    bbu = 'BBU_20_2.0'
    if bbl in out.columns and bbu in out.columns:
        bb_rng = (out[bbu] - out[bbl]).replace(0, np.nan)
        out['close_pos_in_bbands'] = (out['CLOSE'] - out[bbl]) / bb_rng
    stoch = ta.stoch(out['HIGH'], out['LOW'], out['CLOSE'])
    if stoch is not None:
        out = pd.concat([out, stoch], axis=1)
    out['obv'] = ta.obv(out['CLOSE'], out['VOL'])
    out['price_vs_ema_10'] = out['CLOSE'] / out['ema_10'] - 1.0
    out['price_vs_ema_20'] = out['CLOSE'] / out['ema_20'] - 1.0
    out['price_vs_sma_20'] = out['CLOSE'] / out['sma_20'] - 1.0
    return out

def add_target(df):
    out = df.copy(deep=True)
    out['next_open'] = out['OPEN'].shift(-1)
    out['next_close'] = out['CLOSE'].shift(-1)
    out['target_is_green_next'] = (out['next_close'] > out['next_open']).astype('int8')
    out = out.drop(columns=['next_open', 'next_close'])
    out = out.iloc[:-1].reset_index(drop=True)
    return out

print('Feature engineering functions defined ✓')

## Cell 4 — Master Pipeline Builder

In [ ]:
def build_feature_dataframe(df_raw):
    df = prepare_ohlcv_dataframe(df_raw)
    df = add_domain_features(df)
    df = add_rolling_features(df, windows=(6, 12, 24))
    df = add_calendar_features(df)
    df = add_ta_features(df)
    df = add_target(df)
    return df

def get_leakage_columns(df):
    leakage = ['datetime', 'DATE', 'TIME', 'TICKER', 'PER', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL']
    target_cols = [c for c in df.columns if c.startswith('target_')]
    return list(set(leakage + target_cols))

def build_X_y_for_model(feature_df, n_in=5):
    leakage_cols = get_leakage_columns(feature_df)
    feature_only_cols = [c for c in feature_df.columns if c not in leakage_cols]
    X_base = feature_df[feature_only_cols].copy()
    y_full = feature_df['target_is_green_next'].copy()
    valid_idx = X_base.dropna().index
    X_clean = X_base.loc[valid_idx].reset_index(drop=True)
    y_clean = y_full.loc[valid_idx].reset_index(drop=True)
    X_supervised = series_to_supervised(X_clean, n_in=n_in, n_out=1, dropnan=True)
    y_aligned = y_clean.iloc[n_in:].reset_index(drop=True)
    assert len(X_supervised) == len(y_aligned)
    return X_supervised, y_aligned, leakage_cols

print('Pipeline builder functions defined ✓')

## Cell 5 — Load Data & Build Features

In [ ]:
DATA_PATH = './Сбербанк/year_result.csv'
frame = pd.read_csv(DATA_PATH, header=0, sep=';')
frame.columns = [c.strip('<>').strip() for c in frame.columns]
print(f'Загружено строк: {len(frame):,}')
print(f'Колонки: {frame.columns.tolist()}')
frame.head(3)

In [ ]:
feature_df = build_feature_dataframe(frame)
print(f'Итоговый DataFrame: {feature_df.shape[0]:,} строк × {feature_df.shape[1]} колонок')
print(feature_df['target_is_green_next'].value_counts(normalize=True).rename({0: 'red (0)', 1: 'green (1)'}).round(3))
feature_df.head(3)

## Cell 6 — EDA

In [ ]:
eda_cols = ['ret_1', 'body_to_range', 'close_pos_in_range', 'range_pct_close',
            'vol_ratio_6', 'rsi_14', 'close_vs_ma_12']
fig, axes = plt.subplots(1, len(eda_cols), figsize=(20, 4))
for ax, col in zip(axes, eda_cols):
    if col in feature_df.columns:
        feature_df[col].dropna().hist(bins=60, ax=ax, color='steelblue', edgecolor='none', alpha=0.8)
        ax.set_title(col, fontsize=9)
plt.suptitle('Feature distributions (raw)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
top_feat = ['ret_1', 'ret_6', 'ret_12', 'body_to_range', 'close_pos_in_range',
            'vol_ratio_6', 'rsi_14', 'close_vs_ma_12', 'range_zscore_12',
            'is_opening_30m', 'direction', 'target_is_green_next']
top_feat = [c for c in top_feat if c in feature_df.columns]
plt.figure(figsize=(12, 9))
sns.heatmap(feature_df[top_feat].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Key Features + Target')
plt.tight_layout(); plt.show()

## Cell 7 — Build Supervised Dataset & Time Split

In [ ]:
X, y, leakage_cols = build_X_y_for_model(feature_df, n_in=5)
print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'Баланс классов в y: {y.value_counts(normalize=True).round(3).to_dict()}')

n = len(X)
n_train = int(n * 0.70)
n_valid = int(n * 0.15)
X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
X_valid, y_valid = X.iloc[n_train:n_train+n_valid], y.iloc[n_train:n_train+n_valid]
X_calib, y_calib = X.iloc[n_train+n_valid:], y.iloc[n_train+n_valid:]

for name, Xs, ys in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Calib', X_calib, y_calib)]:
    bal = ys.value_counts(normalize=True).round(3).to_dict()
    print(f'{name:6s}: {len(Xs):6,} rows | class balance: {bal}')

## Cell 8 — XGBoost Classifier

In [ ]:
# ── Вспомогательная функция оценки модели ────────────────────────────────
def evaluate_model(model, X_tr, y_tr, X_va, y_va, X_ca, y_ca, model_name='Model'):
    results = {}
    for split_name, Xs, ys in [('Train', X_tr, y_tr), ('Valid', X_va, y_va), ('Calib', X_ca, y_ca)]:
        preds = model.predict(Xs)
        probas = model.predict_proba(Xs)[:, 1]
        acc = accuracy_score(ys, preds)
        auc = roc_auc_score(ys, probas)
        ll = log_loss(ys, probas)
        brier = brier_score_loss(ys, probas)
        results[split_name] = {'accuracy': acc, 'roc_auc': auc, 'log_loss': ll, 'brier': brier}
        print(f'[{model_name}] {split_name:6s} | Acc={acc:.4f} AUC={auc:.4f} LogLoss={ll:.4f} Brier={brier:.4f}')
    return results

# ── XGBoost: не требует StandardScaler — деревья инвариантны к масштабу ──
_neg = int((y_train == 0).sum())
_pos = int((y_train == 1).sum())
_scale_pos_weight = _neg / _pos if _pos > 0 else 1.0
print(f'Class balance → neg={_neg:,}, pos={_pos:,}, scale_pos_weight={_scale_pos_weight:.4f}')

baseline_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=_scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

# eval_set используется ТОЛЬКО для early stopping, не для обучения весов
baseline_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print(f'Best iteration: {baseline_model.best_iteration}')

print('\n=== Baseline: XGBoost ===')
baseline_results = evaluate_model(baseline_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'XGB')

In [ ]:
print('\n=== Classification Report (Valid) ===')
print(classification_report(y_valid, baseline_model.predict(X_valid), target_names=['red (0)', 'green (1)']))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_valid, baseline_model.predict(X_valid))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Red', 'Pred Green'],
            yticklabels=['Actual Red', 'Actual Green'])
plt.title('Confusion Matrix — XGBoost (Valid)')
plt.tight_layout(); plt.show()

## Cell 9 — Probability Calibration (Platt Scaling)

In [ ]:
# _CALIB_ESTIMATOR_KWARG определён в Cell 1 — совместим со sklearn < 1.2 и >= 1.2
calibrated_model = CalibratedClassifierCV(
    **{_CALIB_ESTIMATOR_KWARG: baseline_model},
    method='sigmoid',
    cv='prefit',
)
calibrated_model.fit(X_calib, y_calib)

print('=== Calibrated XGBoost ===')
calib_results = evaluate_model(calibrated_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'XGB+Calib')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for model, label, color in [
    (baseline_model, 'Baseline XGB', 'steelblue'),
    (calibrated_model, 'XGB + Platt', 'tomato'),
]:
    prob_true, prob_pred = calibration_curve(y_valid, model.predict_proba(X_valid)[:, 1], n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', label=label, color=color)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve (Valid)'); ax.legend()
axes[1].hist(baseline_model.predict_proba(X_valid)[:, 1], bins=40, alpha=0.6, label='Baseline XGB', color='steelblue')
axes[1].hist(calibrated_model.predict_proba(X_valid)[:, 1], bins=40, alpha=0.6, label='XGB + Platt', color='tomato')
axes[1].set_xlabel('Predicted probability'); axes[1].set_title('Probability Distribution (Valid)'); axes[1].legend()
plt.tight_layout(); plt.show()

## Cell 10 — Metrics Summary Table

In [ ]:
# ── Сводная таблица метрик: XGBoost vs Calibrated ────────────────────────
rows = []
for split in ['Train', 'Valid', 'Calib']:
    r_base = baseline_results[split]
    r_calib = calib_results[split]
    rows.append({
        'Split': split,
        'XGB Accuracy': round(r_base['accuracy'], 4),
        'XGB AUC': round(r_base['roc_auc'], 4),
        'XGB LogLoss': round(r_base['log_loss'], 4),
        'XGB Brier': round(r_base['brier'], 4),
        'Cal Accuracy': round(r_calib['accuracy'],4),
        'Cal AUC': round(r_calib['roc_auc'], 4),
        'Cal LogLoss': round(r_calib['log_loss'],4),
        'Cal Brier': round(r_calib['brier'], 4),
    })

metrics_df = pd.DataFrame(rows).set_index('Split')

print('\n=== Metrics Summary: XGBoost vs Calibrated ===')
display(
    metrics_df.style
    .format(precision=4)
    .highlight_min(subset=['XGB LogLoss', 'XGB Brier', 'Cal LogLoss', 'Cal Brier'], color='#d4edda')
    .highlight_max(subset=['XGB Accuracy', 'XGB AUC', 'Cal Accuracy', 'Cal AUC'], color='#d4edda')
    .set_caption('Green = best value per column | Lower is better for LogLoss/Brier | Higher is better for Accuracy/AUC')
)

## Cell 11 — Feature Importance

In [ ]:
TOP_N = 30

# ── 1. Built-in importance (gain) ────────────────────────────────────────
importance_gain = (
    pd.Series(baseline_model.get_booster().get_score(importance_type='gain'))
    .sort_values(ascending=False)
    .head(TOP_N)
)

# ── 2. Permutation importance на Valid ───────────────────────────────────
# n_repeats=10 — усредняем для устойчивости; scoring='roc_auc' — целевая метрика.
# Считаем на X_valid, который модель не видела во время обучения весов.
perm_result = permutation_importance(
    baseline_model, X_valid, y_valid,
    n_repeats=10, scoring='roc_auc', random_state=42, n_jobs=-1,
)
perm_df = (
    pd.DataFrame({'feature': X_valid.columns,
                  'importance': perm_result.importances_mean,
                  'std': perm_result.importances_std})
    .sort_values('importance', ascending=False)
    .head(TOP_N)
)

# ── Визуализация ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 11))

importance_gain.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title(f'XGBoost Feature Importance — Gain (top {TOP_N})', fontsize=13)
axes[0].set_xlabel('Mean Gain')
axes[0].axvline(importance_gain.mean(), color='tomato', linestyle='--', lw=1.5, label='Mean')
axes[0].legend()

perm_plot = perm_df.sort_values('importance')
axes[1].barh(perm_plot['feature'], perm_plot['importance'], xerr=perm_plot['std'],
             color='seagreen', edgecolor='none', alpha=0.85, capsize=3)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title(f'Permutation Importance (AUC, Valid, top {TOP_N})', fontsize=13)
axes[1].set_xlabel('Mean AUC decrease when feature permuted')

plt.suptitle('XGBoost Feature Importance Analysis', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

# ── Сводная таблица топ-20 ────────────────────────────────────────────────
print('\n=== Top-20 Features by Permutation Importance (Valid AUC) ===')
display(
    perm_df.head(20)
    .assign(importance=lambda d: d['importance'].round(6),
            std=lambda d: d['std'].round(6))
    .reset_index(drop=True)
    .style.bar(subset=['importance'], color='#5fba7d', vmin=0)
    .format({'importance': '{:.6f}', 'std': '±{:.6f}'})
    .set_caption('Permutation Importance: mean AUC drop on Valid set (n_repeats=10)')
)

# ── Кандидаты на удаление ────────────────────────────────────────────────
zero_perm = perm_df[perm_df['importance'] <= 0]['feature'].tolist()
if zero_perm:
    print(f'\n⚠️ Признаки с importance ≤ 0 (кандидаты на удаление): {len(zero_perm)}')
    print(zero_perm)
else:
    print('\n✓ Все признаки имеют положительную permutation importance.')

## Next Steps

- [ ] Walk-forward validation (expanding window)
- [ ] Убрать признаки с permutation importance ≤ 0 → переобучить
- [ ] Hyperparameter tuning (Optuna / Bayesian search)
- [ ] Симуляция транзакционных издержек
- [ ] Regime detection (rolling volatility / HMM)
- [ ] Meta-labeling для фильтрации сигналов